# Devoir — Machine Learning sous Python
## Stratégie de trading sur l'indice CAC 40 (Feedforward Neural Network)

**MBA2 Trading et Finance de Marché — A25 — Prof. H. SUN**

**Nom Prénom :** *(à compléter avant le dépôt — renommer le fichier en `Nom_Prenom.ipynb`)*

---

### Objectif
Construire une stratégie de trading à l'aide d'un réseau de neurones *feedforward* (apprentissage
supervisé) qui prévoit la **hausse du cours du CAC 40 à J+1**, puis **backtester** cette stratégie
sur la période de test.

### Démarche (les 10 étapes du sujet)
1. Téléchargement des données journalières `^FCHI` (01/01/2017 → 01/01/2022).
2. Construction des *features* : variations de `Close` (J-1 et J-8), ADX(20), RSI(20),
   Stochastique (14, 3), CCI(20), volatilité 10 jours.
3. Construction du *label* : 1 si hausse à J+1 **et** variation `Close` vs J-8 positive, 0 sinon.
4. Séparation *train / test* (temporelle, sans mélange) et *StandardScaling*.
5. & 6. Définition et compilation du réseau (512 elu, 256 relu, 128 elu, 32 relu, dropout 15 %, sortie sigmoïde).
7. Entraînement (`batch_size=64`, `epochs=25`).
8. Prédiction de la hausse à J+1 sur les données de test.
9. Qualité de la prédiction : matrice de confusion + rapport de classification.
10. Backtest de la stratégie : bilan, statistiques et rendement cumulé.


### Installation des dépendances (Google Colab)
**À exécuter en premier**, une fois par session Colab (installe `TA-Lib`, `yfinance`, etc.).
`TA-Lib` est distribué avec des *wheels* précompilés : `pip install` suffit, pas de compilation.
Si l'environnement possède déjà ces bibliothèques, la cellule se termine simplement sans rien réinstaller.

In [ ]:
%pip install -q yfinance TA-Lib tensorflow scikit-learn matplotlib

### Imports

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import talib
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout

# Reproductibilité
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

## 1) Téléchargement des données CAC 40

Données journalières de l'indice CAC 40 (`^FCHI`) sur Yahoo! Finance, du **1er janvier 2017**
au **1er janvier 2022**. On désactive `auto_adjust` pour conserver les colonnes OHLC brutes.

In [ ]:
df = yf.download("^FCHI", start="2017-01-01", end="2022-01-01",
                 interval="1d", auto_adjust=False, progress=False)

# Depuis yfinance >= 0.2, les colonnes sont un MultiIndex (Prix, Ticker) : on l'aplatit.
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.droplevel(1)

print("Dimensions :", df.shape)
df.head()

## 2) Calcul des *features*

À partir des colonnes OHLC, on déduit les 7 variables explicatives demandées. Toutes ces
grandeurs sont **connues à la clôture du jour t** : elles ne contiennent donc aucune information
future (pas de *look-ahead*).

| Feature | Description |
|---|---|
| `var_close_j1` | Variation en % de `Close` vs J-1 |
| `var_close_j8` | Variation en % de `Close` vs J-8 |
| `ADX` | Average Directional Index, `timeperiod=20` |
| `RSI` | Relative Strength Index, `timeperiod=20` |
| `STOCH_K` | Stochastique lent %K, `fastk_period=14`, `slowk_period=3` |
| `CCI` | Commodity Channel Index, `timeperiod=20` |
| `VOLAT_10` | Volatilité (écart-type) des rendements sur 10 jours |

In [ ]:
# Variations de la clôture
df["var_close_j1"] = df["Close"].pct_change(1) * 100
df["var_close_j8"] = df["Close"].pct_change(8) * 100

# Indicateurs techniques (TA-Lib)
df["ADX"] = talib.ADX(df["High"], df["Low"], df["Close"], timeperiod=20)
df["RSI"] = talib.RSI(df["Close"], timeperiod=20)

slowk, slowd = talib.STOCH(df["High"], df["Low"], df["Close"],
                           fastk_period=14, slowk_period=3, slowd_period=3)
df["STOCH_K"] = slowk

df["CCI"] = talib.CCI(df["High"], df["Low"], df["Close"], timeperiod=20)

# Volatilité 10 jours des rendements quotidiens (en %)
df["VOLAT_10"] = df["Close"].pct_change().rolling(10).std() * 100

FEATURES = ["var_close_j1", "var_close_j8", "ADX", "RSI", "STOCH_K", "CCI", "VOLAT_10"]
df[FEATURES].tail()

## 3) Construction du *label*

Le label vaut **1** lorsque les deux conditions sont réunies :
- le cours de clôture **monte à J+1** (rendement du lendemain strictement positif) ;
- la variation de `Close` vs **J-8** est positive.

Sinon le label vaut **0**. On calcule au passage le rendement du lendemain `ret_next`,
réutilisé pour le backtest.

In [ ]:
# Rendement du lendemain (t -> t+1). Sert au label ET au backtest.
df["ret_next"] = df["Close"].pct_change().shift(-1)

df["Label"] = ((df["ret_next"] > 0) & (df["var_close_j8"] > 0)).astype(int)

print("Répartition des classes :")
print(df["Label"].value_counts())
print("\nProportion de 1 : {:.1%}".format(df["Label"].mean()))

## 4) Séparation *train / test* et normalisation

Les données étant **temporelles**, on découpe **sans mélange** (`shuffle=False`) : les 80 %
les plus anciens servent à l'entraînement, les 20 % les plus récents au test. Le
`StandardScaler` est **ajusté uniquement sur le train** puis appliqué au test, afin d'éviter
toute fuite d'information.

In [ ]:
# On retire les lignes contenant des NaN (démarrage des indicateurs, dernière ligne du shift).
df = df.dropna().copy()

X = df[FEATURES].values
y = df["Label"].values

split = int(len(df) * 0.8)              # découpage temporel 80 / 20
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit sur le train seulement
X_test_scaled = scaler.transform(X_test)

idx_test = df.index[split:]
print("Train :", X_train_scaled.shape, "| Test :", X_test_scaled.shape)
print("Période de test :", idx_test[0].date(), "->", idx_test[-1].date())

## 5) & 6) Définition et compilation du réseau de neurones

Architecture *feedforward* demandée :

- **Input Layer** : 7 entrées (les 7 features)
- **Hidden Layer 1** : 512 neurones — activation `elu`
- **Hidden Layer 2** : 256 neurones — activation `relu`
- **Hidden Layer 3** : 128 neurones — activation `elu`
- **Hidden Layer 4** : 32 neurones — activation `relu`
- **Output Layer** : 1 neurone — activation `sigmoid` (classification binaire)
- **Dropout de 15 %** après chaque couche cachée pour limiter le sur-apprentissage.

Compilation : perte `binary_crossentropy`, optimiseur `adam`, métrique `accuracy`.

In [ ]:
model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(512, activation="elu"),
    Dropout(0.15),
    Dense(256, activation="relu"),
    Dropout(0.15),
    Dense(128, activation="elu"),
    Dropout(0.15),
    Dense(32, activation="relu"),
    Dropout(0.15),
    Dense(1, activation="sigmoid"),
])

model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

## 7) Entraînement du réseau

Entraînement sur les données de *training* avec `batch_size=64` et `epochs=25`.
On conserve l'historique pour tracer les courbes d'apprentissage.

In [ ]:
history = model.fit(X_train_scaled, y_train,
                    batch_size=64, epochs=25, verbose=1)

In [ ]:
# Courbes d'apprentissage (perte et accuracy sur le training)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(history.history["loss"]);      ax[0].set_title("Loss (train)");     ax[0].set_xlabel("epoch"); ax[0].grid(True)
ax[1].plot(history.history["accuracy"]);  ax[1].set_title("Accuracy (train)"); ax[1].set_xlabel("epoch"); ax[1].grid(True)
plt.tight_layout(); plt.show()

## 8) Prédiction de la hausse à J+1 sur les données de test

Le réseau renvoie une probabilité de hausse. On applique le seuil **0,5** : au-dessus, on prédit
une hausse (classe 1) et donc un signal d'achat pour le lendemain.

In [ ]:
y_proba = model.predict(X_test_scaled, verbose=0).ravel()
y_pred = (y_proba > 0.5).astype(int)

print("Hausses prédites : {} sur {} jours de test".format(int(y_pred.sum()), len(y_pred)))

## 9) Qualité de la prédiction

Comparaison des prédictions au *label* réel via la **matrice de confusion** et le
**rapport de classification** (`precision`, `recall`, `f1-score`).

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print("Matrice de confusion :")
print(cm)
print()
print(classification_report(y_test, y_pred, digits=3, zero_division=0))

# Visualisation de la matrice de confusion
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xlabel("Prédiction"); ax.set_ylabel("Réel")
ax.set_title("Matrice de confusion")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=13)
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

## 10) Backtest de la stratégie

**Règle de trading :** lorsque le modèle prédit une hausse pour le lendemain (signal = 1), on
achète à la clôture du jour *t* et on encaisse le rendement du jour suivant (`ret_next`, du jour
*t* au jour *t+1*). Sinon on reste en cash (rendement 0). Aucune position vendeuse.

On fournit le **bilan** (nombre d'ordres, gagnants/perdants, P&L…), quelques **statistiques**
de risque, ainsi que le **rendement cumulé** sur la période de test, comparé au *buy & hold*.

In [ ]:
bt = df.iloc[split:].copy()
bt["signal"] = y_pred
bt["strat_ret"] = np.where(bt["signal"] == 1, bt["ret_next"], 0.0)

# --- Bilan des ordres ---
n_orders = int((bt["signal"] == 1).sum())
wins = int(((bt["signal"] == 1) & (bt["ret_next"] > 0)).sum())
losses = n_orders - wins
win_rate = 100 * wins / n_orders if n_orders else 0.0
pnl = bt["strat_ret"].sum() * 100                         # somme des rendements (%)
avg_trade = pnl / n_orders if n_orders else 0.0

# --- Rendements cumulés (capitalisés) ---
cum_strat = (1 + bt["strat_ret"]).prod() - 1
cum_bh = (1 + bt["ret_next"]).prod() - 1                  # buy & hold sur la même période

# --- Statistiques de risque de la stratégie ---
equity = (1 + bt["strat_ret"]).cumprod()
max_dd = (equity / equity.cummax() - 1).min()
sharpe = np.sqrt(252) * bt["strat_ret"].mean() / (bt["strat_ret"].std() + 1e-12)

print("================ BILAN DE LA STRATÉGIE ================")
print("Période de test              : {} -> {}".format(idx_test[0].date(), idx_test[-1].date()))
print("Nombre total d'ordres passés : {}".format(n_orders))
print("Ordres gagnants              : {}".format(wins))
print("Ordres perdants              : {}".format(losses))
print("Taux de réussite             : {:.1f}%".format(win_rate))
print("P&L (somme des rendements)   : {:.2f}%".format(pnl))
print("Rendement moyen par ordre    : {:.3f}%".format(avg_trade))
print("------------------------------------------------------")
print("Rendement cumulé stratégie   : {:.2f}%".format(cum_strat * 100))
print("Rendement cumulé buy & hold  : {:.2f}%".format(cum_bh * 100))
print("Sharpe annualisé (stratégie) : {:.2f}".format(sharpe))
print("Max drawdown (stratégie)     : {:.2f}%".format(max_dd * 100))
print("======================================================")

In [ ]:
# Rendement cumulé de la stratégie vs buy & hold sur la période de test
cum_strat_curve = (1 + bt["strat_ret"]).cumprod() - 1
cum_bh_curve = (1 + bt["ret_next"]).cumprod() - 1

plt.figure(figsize=(12, 6))
plt.plot(bt.index, cum_strat_curve * 100, label="Stratégie ML", linewidth=2)
plt.plot(bt.index, cum_bh_curve * 100, label="Buy & Hold CAC 40", linewidth=2, alpha=0.8)
plt.title("Rendement cumulé sur la période de test")
plt.ylabel("Rendement cumulé (%)"); plt.xlabel("Date")
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

## Conclusion

Le réseau de neurones *feedforward* prévoit la hausse à J+1 du CAC 40 à partir de 7 indicateurs
techniques. La qualité de la classification est mesurée par la matrice de confusion et le rapport
de classification (étape 9), et la valeur économique du signal est évaluée par le backtest
(étape 10) : bilan des ordres, statistiques de risque (Sharpe, max drawdown) et comparaison du
rendement cumulé de la stratégie au *buy & hold*.

> **Remarques méthodologiques**
> - Le découpage *train/test* est **temporel** et le `StandardScaler` est ajusté **sur le train
>   seulement** : il n'y a pas de fuite d'information.
> - Les *features* n'utilisent que l'information disponible à la clôture du jour *t* ; le rendement
>   futur `ret_next` n'apparaît que dans le *label* (cible) et dans le calcul du P&L, jamais en entrée.
> - Le backtest ne tient pas compte des frais de transaction ni du slippage ; il constitue donc une
>   borne haute de la performance réalisable.
